### 3.2. Generation dataset

This cleaning process includes:
1. The aggregation of the five files containing TERNA data about generated electricity so that we have the complete time series from 2021 to 2025;
2. The deletetion of rows with NULL values, due to the final message that occupies only 1 or 2 cells on each Excel file, indicating which filter had been applied while manually downloading the data;
3. The assignment of the correct data type to each column;
4. Mtu (Market Time Unit) harmonization, since from October 1st energy market is detailed on quartely basis.  

After data cleaning, the new dataset is expected to be copied in the <code>/dataset/clean</code> folder.


In [1]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Future Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

# define initial variables
dataset_path = Path('../dataset/raw')           # input folder path
output_path = Path('../dataset/clean')          # output folder path
output_path.mkdir(parents=True, exist_ok=True)  # create folder if non-existent

YEARS = range(2021, 2026)

In [2]:
# 1. Aggregate generation data in dataframe
gen_dfs = [
    pd.read_excel(dataset_path / f'TERNA-gen-{year}.xlsx')
    for year in YEARS
]
generation_data = pd.concat(gen_dfs, ignore_index=True)

# fast check
print(f'''
----Rows x Cols:----
{generation_data.shape}\n
----First rows:----
{generation_data.head()}\n
----Last rows:----
{generation_data.tail()}
''')


----Rows x Cols:----
(420593, 3)

----First rows:----
                  Date  Actual Generation    Primary Source
0  2021-12-31 23:00:00             11.420           Thermal
1  2021-12-31 23:00:00              3.460             Hydro
2  2021-12-31 23:00:00              0.640        Geothermal
3  2021-12-31 23:00:00              0.000      Photovoltaic
4  2021-12-31 23:00:00              1.516  Self-consumption

----Last rows:----
                                   Date  Actual Generation Primary Source
420588              2025-01-01 00:00:00               0.60     Geothermal
420589              2025-01-01 00:00:00               0.40           Wind
420590              2025-01-01 00:00:00              14.20        Thermal
420591              2025-01-01 00:00:00               2.68          Hydro
420592  Applied filters:  Year is 2025;                NaN            NaN



In [3]:
print(f'''
{generation_data.info()}\n
----Date description----
{generation_data['Date'].describe()}\n
----Actual energy generation description----
{generation_data['Actual Generation'].describe()}\n
----Source of production----
{generation_data['Primary Source'].describe()}\n
''')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420593 entries, 0 to 420592
Data columns (total 3 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Date               420593 non-null  object 
 1   Actual Generation  420588 non-null  float64
 2   Primary Source     420588 non-null  object 
dtypes: float64(1), object(2)
memory usage: 9.6+ MB

None

----Date description----
count                  420593
unique                  70098
top       2024-10-27 02:00:00
freq                       12
Name: Date, dtype: object

----Actual energy generation description----
count    420588.000000
mean          5.083124
std           6.004252
min          -0.010000
25%           0.630000
50%           2.960000
75%           6.560000
max          35.500000
Name: Actual Generation, dtype: float64

----Source of production----
count      420588
unique          6
top       Thermal
freq        70098
Name: Primary Source, dtype: object




In [4]:
# 2. Delete n=5 Rows with NULL values it's the row at the end of each dataset (for each year) which explains the applied filter
generation_data = generation_data.dropna(axis=0, how='any')
print(generation_data.head(10) )       # check

# 3. Convert Date from Object type to Datetime type
generation_data['Date'] = pd.to_datetime(generation_data['Date'], dayfirst=True, errors='coerce')
#    Convert Primary Source from Object to String
generation_data['Primary Source'] = generation_data['Primary Source'].astype('string')

generation_data.columns

                  Date  Actual Generation    Primary Source
0  2021-12-31 23:00:00             11.420           Thermal
1  2021-12-31 23:00:00              3.460             Hydro
2  2021-12-31 23:00:00              0.640        Geothermal
3  2021-12-31 23:00:00              0.000      Photovoltaic
4  2021-12-31 23:00:00              1.516  Self-consumption
5  2021-12-31 23:00:00              1.820              Wind
6  2021-12-31 22:00:00              3.720             Hydro
7  2021-12-31 22:00:00              1.476  Self-consumption
8  2021-12-31 22:00:00              0.000      Photovoltaic
9  2021-12-31 22:00:00              0.640        Geothermal


Index(['Date', 'Actual Generation', 'Primary Source'], dtype='object')

In [5]:
# 4. MTU harmonization
generation_data = (
    generation_data
    .groupby(['Primary Source', pd.Grouper(key='Date', freq='h')])
    .sum()
    .reset_index()
)

# pivoted table
generation_pivot = generation_data.pivot(
    index='Date',
    columns='Primary Source',
    values='Actual Generation'
)

It is now useful to add new columns for further data analysis. 
<code>TOT Generation</code> sums the total generated energy that entered in the energy market.  
It sums <code>Geothermal</code>, <code>Hydro</code>, <code>Photovoltaic</code>, <code>Thermal</code>, <code>Wind</code>.  
<code>Self-consumption</code> is excluded by this calculation, since it represents the amount of energy produced, but employed for self consumption and so, hasn't been traded in energy markets.

In [6]:
# calculate total generation 
generation_pivot['TOT Generation'] = generation_pivot[['Geothermal', 'Hydro', 'Photovoltaic', 'Thermal', 'Wind',]].sum(axis=1)
generation_pivot.head()

Primary Source,Geothermal,Hydro,Photovoltaic,Self-consumption,Thermal,Wind,TOT Generation
Date,,,,,,,
2021-01-01 00:00:00,0.62,3.43,0.0,2.058,14.55,2.60,21.20
2021-01-01 01:00:00,0.62,3.03,0.0,1.952,13.68,2.75,20.08
2021-01-01 02:00:00,0.62,2.79,0.0,1.812,12.77,2.63,18.81
2021-01-01 03:00:00,0.62,2.60,0.0,1.735,12.12,2.62,17.96
2021-01-01 04:00:00,0.62,2.60,0.0,1.733,12.06,2.61,17.89


<code>Renewables weight</code> is an index calculated by the total of energy produced by renewables, divided by the total amount of energy generated. In this case <code>Thermal</code> energy is excluded by this list, since it represents the coverage through fossils.

In [9]:
# add renewables weights column
generation_pivot['Renewables weight'] = (
    generation_pivot[['Geothermal', 'Hydro', 'Photovoltaic', 'Wind']].sum(axis=1)
    / generation_pivot['TOT Generation']
)

# export dataset
generation_pivot.to_csv(output_path / 'gen-clean.csv')

# show updated table
generation_pivot

Primary Source,Geothermal,Hydro,Photovoltaic,Self-consumption,Thermal,Wind,TOT Generation,Renewables weight
Date,,,,,,,,
2021-01-01 00:00:00,0.62,3.43,0.0,2.058,14.55,2.60,21.20,0.313679
2021-01-01 01:00:00,0.62,3.03,0.0,1.952,13.68,2.75,20.08,0.318725
2021-01-01 02:00:00,0.62,2.79,0.0,1.812,12.77,2.63,18.81,0.321106
2021-01-01 03:00:00,0.62,2.60,0.0,1.735,12.12,2.62,17.96,0.325167
2021-01-01 04:00:00,0.62,2.60,0.0,1.733,12.06,2.61,17.89,0.325880
...,...,...,...,...,...,...,...,...
2025-12-31 19:00:00,2.40,15.00,0.0,13.676,90.60,9.32,117.32,0.227753
2025-12-31 20:00:00,2.40,11.28,0.0,12.580,85.32,8.32,107.32,0.204994
2025-12-31 21:00:00,2.40,8.44,0.0,10.764,80.44,6.68,97.96,0.178849


Back: [3.1. Prices data cleaning](3.1_prices-data-cleaning.ipynb)  

Next: [3.3. Energy load data cleaning](3.3_load-data-cleaning.ipynb)
